# Simple Market Pricers - Examples and Analysis

This notebook demonstrates the usage of the Simple Market Pricers library for pricing:
- European options using Black-Scholes model
- Fixed-coupon bonds with yield-to-maturity calculation
- Plain-vanilla interest rate swaps

## Table of Contents
1. [Option Pricing Examples](#option-pricing)
2. [Bond Pricing Examples](#bond-pricing)
3. [Interest Rate Swap Examples](#swap-pricing)
4. [Sensitivity Analysis](#sensitivity-analysis)
5. [Performance Notes](#performance-notes)


In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pricers import (
    OptionPricer, BondPricer, SwapPricer, MarketData,
    price_option, price_bond, price_swap
)

# Set up plotting style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Simple Market Pricers Library Loaded Successfully!")
print("=" * 50)


## 1. Option Pricing Examples {#option-pricing}

Let's start with some basic option pricing examples using the Black-Scholes model.


In [ ]:
# Basic option pricing example
S = 100.0    # Current stock price
K = 100.0    # Strike price (at-the-money)
T = 0.25     # Time to expiration (3 months)
r = 0.05     # Risk-free rate (5%)
sigma = 0.20 # Volatility (20%)

# Price call and put options
call_result = price_option('call', S, K, T, r, sigma)
put_result = price_option('put', S, K, T, r, sigma)

print("Option Pricing Results:")
print(f"Stock Price: ${S}")
print(f"Strike Price: ${K}")
print(f"Time to Expiration: {T} years ({T*12} months)")
print(f"Risk-free Rate: {r*100:.1f}%")
print(f"Volatility: {sigma*100:.1f}%")
print("-" * 40)
print(f"Call Option Price: ${call_result['price']:.2f}")
print(f"Put Option Price: ${put_result['price']:.2f}")
print("-" * 40)
print("Call Greeks:")
print(f"  Delta: {call_result['delta']:.3f}")
print(f"  Gamma: {call_result['gamma']:.3f}")
print(f"  Theta: {call_result['theta']:.3f} (daily)")
print(f"  Vega: {call_result['vega']:.3f} (per 1% vol)")
print("-" * 40)
print("Put Greeks:")
print(f"  Delta: {put_result['delta']:.3f}")
print(f"  Gamma: {put_result['gamma']:.3f}")
print(f"  Theta: {put_result['theta']:.3f} (daily)")
print(f"  Vega: {put_result['vega']:.3f} (per 1% vol)")


In [ ]:
# Option price vs. volatility analysis
volatilities = np.linspace(0.05, 0.50, 20)
call_prices = []
put_prices = []
deltas = []

for vol in volatilities:
    call_result = price_option('call', S, K, T, r, vol)
    put_result = price_option('put', S, K, T, r, vol)
    call_prices.append(call_result['price'])
    put_prices.append(put_result['price'])
    deltas.append(call_result['delta'])

# Create plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Option prices vs volatility
ax1.plot(volatilities * 100, call_prices, 'b-', linewidth=2, label='Call Option')
ax1.plot(volatilities * 100, put_prices, 'r-', linewidth=2, label='Put Option')
ax1.set_xlabel('Volatility (%)')
ax1.set_ylabel('Option Price ($)')
ax1.set_title('Option Prices vs Volatility')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Delta vs volatility
ax2.plot(volatilities * 100, deltas, 'g-', linewidth=2)
ax2.set_xlabel('Volatility (%)')
ax2.set_ylabel('Call Delta')
ax2.set_title('Call Delta vs Volatility')
ax2.grid(True, alpha=0.3)

# Plot 3: Option prices vs stock price
stock_prices = np.linspace(80, 120, 50)
call_prices_sp = []
put_prices_sp = []

for sp in stock_prices:
    call_result = price_option('call', sp, K, T, r, sigma)
    put_result = price_option('put', sp, K, T, r, sigma)
    call_prices_sp.append(call_result['price'])
    put_prices_sp.append(put_result['price'])

ax3.plot(stock_prices, call_prices_sp, 'b-', linewidth=2, label='Call Option')
ax3.plot(stock_prices, put_prices_sp, 'r-', linewidth=2, label='Put Option')
ax3.axvline(x=K, color='k', linestyle='--', alpha=0.5, label='Strike Price')
ax3.set_xlabel('Stock Price ($)')
ax3.set_ylabel('Option Price ($)')
ax3.set_title('Option Prices vs Stock Price')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Time decay
times = np.linspace(0.01, 1.0, 50)
call_prices_t = []
put_prices_t = []

for t in times:
    call_result = price_option('call', S, K, t, r, sigma)
    put_result = price_option('put', S, K, t, r, sigma)
    call_prices_t.append(call_result['price'])
    put_prices_t.append(put_result['price'])

ax4.plot(times * 12, call_prices_t, 'b-', linewidth=2, label='Call Option')
ax4.plot(times * 12, put_prices_t, 'r-', linewidth=2, label='Put Option')
ax4.set_xlabel('Time to Expiration (months)')
ax4.set_ylabel('Option Price ($)')
ax4.set_title('Option Prices vs Time to Expiration')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 2. Bond Pricing Examples {#bond-pricing}

Now let's explore bond pricing and yield-to-maturity calculations.


In [ ]:
# Basic bond pricing example
face_value = 1000.0      # Bond face value
coupon_rate = 0.05       # 5% annual coupon
years_to_maturity = 5.0  # 5 years to maturity
yield_rate = 0.04        # 4% market yield
payments_per_year = 2    # Semi-annual payments

# Price the bond
bond_result = price_bond(face_value, coupon_rate, years_to_maturity, yield_rate, payments_per_year)

print("Bond Pricing Results:")
print(f"Face Value: ${face_value}")
print(f"Coupon Rate: {coupon_rate*100:.1f}%")
print(f"Years to Maturity: {years_to_maturity}")
print(f"Market Yield: {yield_rate*100:.1f}%")
print(f"Payments per Year: {payments_per_year}")
print("-" * 40)
print(f"Bond Price: ${bond_result['price']:.2f}")
print(f"Macaulay Duration: {bond_result['duration']:.2f} years")
print("-" * 40)

# Calculate YTM from the price
ytm = BondPricer.yield_to_maturity(face_value, coupon_rate, years_to_maturity, 
                                  bond_result['price'], payments_per_year)
print(f"YTM from Price: {ytm*100:.4f}%")
print(f"Original Yield: {yield_rate*100:.4f}%")
print(f"Difference: {(ytm-yield_rate)*100:.4f}%")


In [ ]:
# Bond price vs yield analysis
yields = np.linspace(0.01, 0.10, 30)
bond_prices = []
durations = []

for y in yields:
    result = price_bond(face_value, coupon_rate, years_to_maturity, y, payments_per_year)
    bond_prices.append(result['price'])
    durations.append(result['duration'])

# Create plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Bond price vs yield
ax1.plot(yields * 100, bond_prices, 'b-', linewidth=2)
ax1.axhline(y=face_value, color='r', linestyle='--', alpha=0.7, label='Par Value')
ax1.axvline(x=coupon_rate * 100, color='g', linestyle='--', alpha=0.7, label='Coupon Rate')
ax1.set_xlabel('Yield (%)')
ax1.set_ylabel('Bond Price ($)')
ax1.set_title('Bond Price vs Yield (Convexity)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Duration vs yield
ax2.plot(yields * 100, durations, 'g-', linewidth=2)
ax2.set_xlabel('Yield (%)')
ax2.set_ylabel('Duration (years)')
ax2.set_title('Bond Duration vs Yield')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare different bond types
print("\nBond Comparison:")
print("-" * 50)
bond_types = [
    ("Premium Bond", 0.06, 0.04),  # High coupon, low yield
    ("Par Bond", 0.05, 0.05),      # Coupon = yield
    ("Discount Bond", 0.04, 0.06)  # Low coupon, high yield
]

for name, coupon, yield_rate in bond_types:
    result = price_bond(face_value, coupon, years_to_maturity, yield_rate, payments_per_year)
    print(f"{name}:")
    print(f"  Coupon: {coupon*100:.1f}%, Yield: {yield_rate*100:.1f}%")
    print(f"  Price: ${result['price']:.2f}, Duration: {result['duration']:.2f} years")
    print()


## 3. Interest Rate Swap Examples {#swap-pricing}

Let's explore interest rate swap pricing and par swap rate calculations.


In [ ]:
# Basic swap pricing example
notional = 1000000.0     # $1M notional
fixed_rate = 0.05        # 5% fixed rate
floating_rate = 0.04     # 4% current floating rate
years_to_maturity = 2.0  # 2 years to maturity
payments_per_year = 2    # Semi-annual payments

# Price the swap
swap_result = price_swap(notional, fixed_rate, floating_rate, years_to_maturity, payments_per_year)

print("Interest Rate Swap Results:")
print(f"Notional Amount: ${notional:,.0f}")
print(f"Fixed Rate: {fixed_rate*100:.1f}%")
print(f"Floating Rate: {floating_rate*100:.1f}%")
print(f"Years to Maturity: {years_to_maturity}")
print(f"Payments per Year: {payments_per_year}")
print("-" * 50)
print(f"Swap Present Value: ${swap_result['present_value']:,.2f}")
print(f"Par Swap Rate: {swap_result['par_swap_rate']*100:.4f}%")
print("-" * 50)

# Interpretation
if swap_result['present_value'] > 0:
    print("Fixed rate payer receives money (swap is in-the-money)")
else:
    print("Fixed rate payer pays money (swap is out-of-the-money)")

print(f"Par swap rate is {swap_result['par_swap_rate']*100:.4f}% - this is the rate")
print("that would make the swap have zero present value.")


In [ ]:
# Swap analysis: PV vs fixed rate and par swap rate curve
fixed_rates = np.linspace(0.02, 0.08, 30)
swap_pvs = []
par_rates = []

for fr in fixed_rates:
    result = price_swap(notional, fr, floating_rate, years_to_maturity, payments_per_year)
    swap_pvs.append(result['present_value'])
    par_rates.append(result['par_swap_rate'])

# Create plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Swap PV vs fixed rate
ax1.plot(fixed_rates * 100, swap_pvs, 'b-', linewidth=2)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax1.axvline(x=floating_rate * 100, color='r', linestyle='--', alpha=0.7, label='Floating Rate')
ax1.set_xlabel('Fixed Rate (%)')
ax1.set_ylabel('Swap Present Value ($)')
ax1.set_title('Swap PV vs Fixed Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Par swap rate vs maturity
maturities = np.linspace(0.5, 10, 20)
par_rates_maturity = []

for maturity in maturities:
    par_rate = SwapPricer.par_swap_rate(floating_rate, maturity, payments_per_year)
    par_rates_maturity.append(par_rate)

ax2.plot(maturities, [pr * 100 for pr in par_rates_maturity], 'g-', linewidth=2)
ax2.axhline(y=floating_rate * 100, color='r', linestyle='--', alpha=0.7, label='Floating Rate')
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Par Swap Rate (%)')
ax2.set_title('Par Swap Rate vs Maturity')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Discount factor analysis
print("\nDiscount Factor Analysis:")
print("-" * 40)
times = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
for t in times:
    df = SwapPricer.discount_factor(floating_rate, t)
    print(f"Time: {t:4.1f} years, Discount Factor: {df:.4f}")


## 4. Sensitivity Analysis {#sensitivity-analysis}

Let's analyze how sensitive our pricing models are to different parameters.


In [ ]:
# Sensitivity analysis for all instruments
print("SENSITIVITY ANALYSIS")
print("=" * 60)

# Option sensitivity to parameters
print("\n1. OPTION SENSITIVITY:")
print("-" * 30)

# Base case
base_call = price_option('call', 100, 100, 0.25, 0.05, 0.20)
print(f"Base Call Price: ${base_call['price']:.2f}")

# Sensitivity to stock price
call_up = price_option('call', 105, 100, 0.25, 0.05, 0.20)
call_down = price_option('call', 95, 100, 0.25, 0.05, 0.20)
print(f"Stock +5%: ${call_up['price']:.2f} (Δ: ${call_up['price']-base_call['price']:.2f})")
print(f"Stock -5%: ${call_down['price']:.2f} (Δ: ${call_down['price']-base_call['price']:.2f})")

# Sensitivity to volatility
call_vol_up = price_option('call', 100, 100, 0.25, 0.05, 0.25)
call_vol_down = price_option('call', 100, 100, 0.25, 0.05, 0.15)
print(f"Vol +25%: ${call_vol_up['price']:.2f} (Δ: ${call_vol_up['price']-base_call['price']:.2f})")
print(f"Vol -25%: ${call_vol_down['price']:.2f} (Δ: ${call_vol_down['price']-base_call['price']:.2f})")

# Bond sensitivity
print("\n2. BOND SENSITIVITY:")
print("-" * 30)

base_bond = price_bond(1000, 0.05, 5, 0.04)
print(f"Base Bond Price: ${base_bond['price']:.2f}")

# Sensitivity to yield
bond_yield_up = price_bond(1000, 0.05, 5, 0.045)
bond_yield_down = price_bond(1000, 0.05, 5, 0.035)
print(f"Yield +50bp: ${bond_yield_up['price']:.2f} (Δ: ${bond_yield_up['price']-base_bond['price']:.2f})")
print(f"Yield -50bp: ${bond_yield_down['price']:.2f} (Δ: ${bond_yield_down['price']-base_bond['price']:.2f})")

# Swap sensitivity
print("\n3. SWAP SENSITIVITY:")
print("-" * 30)

base_swap = price_swap(1000000, 0.05, 0.04, 2)
print(f"Base Swap PV: ${base_swap['present_value']:,.2f}")

# Sensitivity to floating rate
swap_rate_up = price_swap(1000000, 0.05, 0.045, 2)
swap_rate_down = price_swap(1000000, 0.05, 0.035, 2)
print(f"Floating +50bp: ${swap_rate_up['present_value']:,.2f} (Δ: ${swap_rate_up['present_value']-base_swap['present_value']:,.2f})")
print(f"Floating -50bp: ${swap_rate_down['present_value']:,.2f} (Δ: ${swap_rate_down['present_value']-base_swap['present_value']:,.2f})")


## 5. Performance Notes {#performance-notes}

### Key Assumptions and Limitations

**Options (Black-Scholes):**
- European options only (no early exercise)
- Constant volatility and risk-free rate
- No dividends
- Log-normal stock price distribution
- No transaction costs or bid-ask spreads

**Bonds:**
- Fixed coupon payments
- Flat yield curve assumption
- No credit risk or default probability
- Standard day count conventions
- No call/put features

**Interest Rate Swaps:**
- Plain-vanilla fixed-for-floating swaps
- Flat yield curve assumption
- No credit risk
- Standard day count conventions
- No optionality or embedded features

### Extensions and Improvements

1. **Yield Curve Bootstrapping:** Implement bootstrapped yield curves from market instruments
2. **Monte Carlo Pricing:** Add Monte Carlo methods for path-dependent options
3. **Day Count Conventions:** Implement various day count conventions (30/360, ACT/ACT, etc.)
4. **Credit Risk:** Add credit spread modeling for corporate bonds
5. **American Options:** Implement binomial tree or finite difference methods
6. **Exotic Options:** Add barrier, Asian, and other exotic option types
7. **Real-time Data:** Integrate with market data feeds
8. **Risk Management:** Add VaR, stress testing, and scenario analysis
